# Load data from s3 to a dataframes

In [0]:
%python

# load csvs into dfs
item_path = "s3://merkle-de-interview-case-study/de/item.csv"
event_path = "s3://merkle-de-interview-case-study/de/event.csv"

item_df = spark.read.option("header", True).csv(item_path)
event_df = spark.read.option("header", True).csv(event_path)

display(item_df)
display(event_df)

### Observations

- The dataset has been transferred with string types as requested.

### Item Data
- There are null values in the `adjective` and `modifier` columns; a rule for handling or populating these could be introduced based on agreement.
- The `id` column contains float values, although it should essentially be of type int.
- The `price` column is of float type, but since it represents monetary values, it could potentially be rounded to 2 decimal places.

### Event Data
- `event.payload` has not been fully transferred; the load process needs to be adjusted so that this field, which is inherently of JSON type, can be properly ingested.
- `user_id` is of float type, while it should naturally be int.
- Other columns are strings.

In [0]:
# itam_df looks nice there is nulls in adjectiv and modifier columns, but for a event data column event.payload hasen't been loaded as a json valid string, so this will help us to load it as a json object
event_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .load(event_path)
)

display(event_df)


In [0]:
# check dataframes shapes
item_shape = (item_df.count(), len(item_df.columns))
event_shape = (event_df.count(), len(event_df.columns))

print(f"item_df: {item_shape[0]} rows, {item_shape[1]} columns")
print(f"event_df: {event_shape[0]} rows, {event_shape[1]} columns")

In [0]:
# check laoded bronze tables
display(spark.sql("SHOW TABLES"))
# check bronze tables schema
display(spark.sql("DESCRIBE EXTENDED bronze.item"))
display(spark.sql("DESCRIBE EXTENDED bronze.event"))
#

In [0]:
from pyspark.sql.functions import col

# Cast and rename columns for bronze.event table
bronze_event_df = spark.table("bronze.events_raw") \
    .withColumn("event_id", col("event_id").cast("string")) \
    .withColumn("event_time", col("event_time").cast("timestamp")) \
    .withColumn("user_id", col("user_id").cast("double").cast("int")) \
    .withColumnRenamed("event.payload", "event_payload")

display(bronze_event_df)

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import StructType, StringType

# Define schema for event_payload JSON
event_payload_schema = StructType() \
    .add("event_name", StringType()) \
    .add("platform", StringType()) \
    .add("parameter_name", StringType()) \
    .add("parameter_value", StringType())

# Parse and flatten event_payload column
bronze_event_flat_df = bronze_event_df.withColumn(
    "event_payload_json", from_json(col("event_payload"), event_payload_schema)
).select(
    "event_id",
    "event_time",
    "user_id",
    "event_payload_json.event_name",
    "event_payload_json.platform",
    "event_payload_json.parameter_name",
    "event_payload_json.parameter_value"
)

display(bronze_event_flat_df)

In [0]:
from pyspark.sql.functions import count, countDistinct
# check count an distinct count of event_id
event_id_counts = bronze_event_flat_df.agg(
    count("event_id").alias("event_id_count"),
    countDistinct("event_id").alias("event_id_distinct_count")
)

display(event_id_counts)

# Group by event_name, platform, parameter_name and check count and distinct count of event_id
event_id_grouped_counts = bronze_event_flat_df.groupBy(
    "event_name", "platform", "parameter_name"
).agg(
    count("event_id").alias("event_id_count"),
    countDistinct("event_id").alias("event_id_distinct_count")
)

display(event_id_grouped_counts)

In [0]:
from pyspark.sql.functions import first

# Pivot parameter_name values to columns, making event_id unique
bronze_event_pivot_df = bronze_event_flat_df.groupBy(
    "event_id", "event_time", "user_id", "event_name", "platform"
).pivot("parameter_name").agg(first("parameter_value"))

display(bronze_event_pivot_df)

In [0]:
from pyspark.sql.functions import count, countDistinct
# check count an distinct count of event_id
event_id_counts_pivoted = bronze_event_pivot_df.agg(
    count("event_id").alias("event_id_count"),
    countDistinct("event_id").alias("event_id_distinct_count")
)

display(event_id_counts_pivoted)

In [0]:
# Save bronze_event_pivot_df to silver_events table
bronze_event_pivot_df.write.format("delta").mode("overwrite").saveAsTable("silver_events")

some aditiional checks over the event data

In [0]:
%sql
SELECT -- COUNT(*) AS cnt -- 265874
  -- CAST(itemviewed_user_id_id AS int) AS viewed_user_id
  DISTINCT item_id, test_assignment, test_id, item_id, viewed_user_id
  -- CAST(item_id AS int) AS item_id
  -- CAST(itemviewed_user_id_id AS int) AS viewed_user_id
  -- test_assignment, test_id
FROM silver_events
WHERE 1=1
  -- AND test_assignment IS NOT NULL
  -- AND viewed_user_id IS NOT NULL
  -- AND test_id IS NOT NULL
-- LIMIT 4

In [0]:
from pyspark.sql.functions import col, year

# additional cast over pivoted columns

bronze_event_pivot_casted_df = bronze_event_pivot_df \
    .withColumn("item_id", col("item_id").cast("int")) \
    .withColumn("test_assignment", col("test_assignment").cast("int")) \
    .withColumn("test_id", col("test_id").cast("int")) \
    .withColumn("viewed_user_id", col("viewed_user_id").cast("int")) \
    .withColumn("year", year(col("event_time")))

display(bronze_event_pivot_casted_df)

## Observations

- `Event` is an excellent candidate for a fact table, although it contains certain categorical columns that could be further transformed into a `Junk Dimension`.
- I choose to keep it as is to avoid introducing unnecessary complexity, while maintaining high observability relative to the input data.

In [0]:
bronze_event_pivot_casted_df.write.format("delta").mode("overwrite").partitionBy("year").saveAsTable("silver.fact_events")

In [0]:
df = spark.sql("""
               SELECT 
                    COUNT(DISTINCT event_id) AS cnt_dist
                    , COUNT(*) AS cnt
               FROM silver.fact_events
          """)
display(df)


In [0]:
df = spark.sql("""
               SELECT 
                    COUNT(DISTINCT id) AS cnt_dist
                    , COUNT(*) AS cnt
                    , COUNT(name) AS cnt_name
                    , COUNT(price) AS cnt_price
               FROM bronze.items_raw
          """)

df1 = spark.sql("""
               SELECT *
               FROM bronze.items_raw
          """)
display(df)
display(df1)

In [0]:
from pyspark.sql.functions import coalesce, lit, col, round, to_timestamp

items_transformed_df = spark.table("bronze.items_raw") \
    .withColumn("adjective", coalesce(col("adjective"), lit("Unknown"))) \
    .withColumn("modifier", coalesce(col("modifier"), lit("Unknown"))) \
    .withColumn("created_at", to_timestamp(col("created_at"))) \
    .withColumn("id", col("id").cast("double").cast("int")) \
    .withColumn("price", round(col("price").cast("double"), 2)) 
    # .withColumn("year", year(col("created_at")))

display(items_transformed_df)
# Save bronze_event_pivot_df to silver_events table
items_transformed_df.write.format("delta").mode("overwrite").saveAsTable("silver.items")

In [0]:
df = spark.sql("""
          SELECT DISTINCT modifier
          FROM silver.dim_items
          WHERE price = 0
          """)
display(df)
# Only for modifier how-to-manual prices is 0

df = spark.sql("""
          SELECT count(*) --262786
          FROM silver.fact_events e
          INNER JOIN silver.dim_items i ON i.id = e.item_id
          WHERE 1=1
            -- AND i.price = 0
            -- AND e.item_id IS NOT NULL
          """)
display(df)